In [47]:
import yaml
from pathlib import Path
import re
from dataclasses import dataclass
from typing import List, Any


In [48]:
@dataclass
class HPRow:
    model: str
    hp_name: str
    hp_type: str
    default: str
    domain: str
    log: str


In [54]:
LATEX_SPECIALS = {
    "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
    "_": r"\_", "{": r"\{", "}": r"\}",
    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    # "\\": r"\textbackslash{}",
}

def latex_escape(s: str) -> str:
    return "".join(LATEX_SPECIALS.get(c, c) for c in s)

def model_name_from_path(path: Path) -> str:
    return re.sub(r"\s+", " ", path.stem.strip())

def as_str(v: Any) -> str:
    if v is None:
        return ""
    if isinstance(v, bool):
        return "true" if v else "false"
    return str(v)

def domain_from_hp(hp: dict) -> str:
    if "choices" in hp and isinstance(hp["choices"], list):
        return "{" + ", ".join(as_str(c) for c in hp["choices"]) + "}"

    lower = hp.get("lower")
    upper = hp.get("upper")
    if lower is not None or upper is not None:
        return f"[{as_str(lower)}, {as_str(upper)}]"
    return ""

def latex_escape_no_bs(s: str) -> str:
    # like latex_escape, but do NOT escape backslashes
    specials = {
        "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
        "_": r"\_", "{": r"\{", "}": r"\}",
        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
        # "\\": r"\textbackslash{}",  # intentionally omitted
    }
    return "".join(specials.get(c, c) for c in s)

def latex_range(s: str) -> str:
    s = latex_escape_no_bs(s)                # 1) escape YAML text
    s = s.replace(", ", ", ")                # normalize
    s = s.replace(",", ",\\linebreak[1] ")   # 2) inject LaTeX (not escaped)
    return s



In [55]:
def parse_yaml_file(path: Path) -> List[HPRow]:
    data = yaml.safe_load(path.read_text())

    if not isinstance(data, dict):
        return []

    hps = data.get("hyperparameters", [])
    model = model_name_from_path(path)

    rows = []
    for hp in hps:
        if not isinstance(hp, dict):
            continue

        name = as_str(hp.get("name")).strip()
        if not name:
            continue

        rows.append(
            HPRow(
                model=model,
                hp_name=name,
                hp_type=as_str(hp.get("type")),
                default=as_str(hp.get("default_value")),
                domain=domain_from_hp(hp),
                log="yes" if hp.get("log", False) else "no",
            )
        )
    return rows


In [ ]:
def to_latex_longtable(rows: list[HPRow]) -> str:
    rows = sorted(rows, key=lambda r: (r.model.lower(), r.hp_name.lower()))

    grouped = {}
    for r in rows:
        grouped.setdefault(r.model, []).append(r)

    def log_symbol(log_str: str) -> str:
        # 'yes'/'no' from parser
        return r"$\checkmark$" if log_str.strip().lower() == "yes" else r"\ding{55}"

    def latex_range(s: str) -> str:
        s = s.replace(", ", ",\\,")
        s = s.replace(",", ",\\linebreak[1] ")
        return latex_escape(s)

    lines = []
    lines.append(r"\large")
    lines.append(r"\setlength{\tabcolsep}{3pt}")
    lines.append(r"\begin{longtable}{l l l c p{5.4cm}}")
    lines.append(r"\caption{Hyperparameter search spaces.}\label{tab:search_spaces}\\")
    lines.append(r"\toprule")
    lines.append(r"Model & Parameter & Type & Log & Range \\")
    lines.append(r"\midrule")
    lines.append(r"\endfirsthead")
    lines.append(r"\toprule")
    lines.append(r"Model & Parameter & Type & Log & Range \\")
    lines.append(r"\midrule")
    lines.append(r"\endhead")

    for model, mrows in grouped.items():
        for i, r in enumerate(mrows):
            model_cell = latex_escape(model) if i == 0 else ""
            lines.append(
                f"{model_cell} & {latex_escape(r.hp_name)} & "
                f"{latex_escape(r.hp_type)} & {log_symbol(r.log)} & "
                f"{latex_range(r.domain)} \\\\"
            )
        lines.append(r"\midrule")

    lines.append(r"\end{longtable}")
    return "\n".join(lines)


In [60]:
folder = Path("../configs/search_spaces")   # ← your folder with yml files

all_rows = []
for p in sorted(folder.glob("*.yml")) + sorted(folder.glob("*.yaml")):
    all_rows.extend(parse_yaml_file(p))

print(f"Found {len(all_rows)} hyperparameters")


Found 116 hyperparameters


In [61]:
tex = to_latex_longtable(all_rows)
print(tex)

\small
\setlength{\tabcolsep}{3pt}
\begin{longtable}{l l l c p{5.4cm}}
\caption{Hyperparameter search spaces.}\label{tab:search_spaces}\\
\toprule
Model & Parameter & Type & Log & Range \\
\midrule
\endfirsthead
\toprule
Model & Parameter & Type & Log & Range \\
\midrule
\endhead
AdaBoost & estimator & categorical & \ding{55} & \{DecisionTreeRegressor,\linebreak[1] \,\linebreak[1] Ridge\} \\
 & learning\_rate & uniform\_float & $\checkmark$ & [0.0001,\linebreak[1] \,\linebreak[1] 2.0] \\
 & loss & categorical & \ding{55} & \{linear,\linebreak[1] \,\linebreak[1] square,\linebreak[1] \,\linebreak[1] exponential\} \\
\midrule
Bagging & bootstrap\_features & categorical & \ding{55} & \{true,\linebreak[1] \,\linebreak[1] false\} \\
 & estimator & categorical & \ding{55} & \{DecisionTreeRegressor,\linebreak[1] \,\linebreak[1] Ridge\} \\
 & max\_features & uniform\_float & \ding{55} & [0.1,\linebreak[1] \,\linebreak[1] 0.5] \\
 & max\_samples & uniform\_float & \ding{55} & [0.1,\linebreak[1] 

# Prepro Config Space

In [74]:
from ConfigSpace import ConfigurationSpace, Categorical, Float, Integer, Constant

cs = ConfigurationSpace()

# Preprocessing Search Space
# cs.add(Constant("enable_numeric_features", value=True))
cs.add(Categorical("enable_categorical_features", [True, False],default=True))
cs.add(Categorical("enable_datetime_features", [True, False],default=True))
cs.add(Categorical("enable_text_special_features", [True, False],default=True))
cs.add(Categorical("enable_text_ngram_features", [True, False],default=True))
cs.add(Categorical("enable_raw_text_features", [True, False],default=False))
cs.add(Categorical("allow_nans", [True, False],default=False))
cs.add(Categorical("prepro_analyzer", ["word", "char", "char_wb"], default="word"))
cs.add(Float("prepro_min_df", (0.005, 1.0), log=True, default=0.2))
cs.add(Integer("prepro_max_features", (5, 20000), log=True, default=10000))
cs

Configuration space object:
  Hyperparameters:
    allow_nans, Type: Categorical, Choices: {True, False}, Default: False
    enable_categorical_features, Type: Categorical, Choices: {True, False}, Default: True
    enable_datetime_features, Type: Categorical, Choices: {True, False}, Default: True
    enable_raw_text_features, Type: Categorical, Choices: {True, False}, Default: False
    enable_text_ngram_features, Type: Categorical, Choices: {True, False}, Default: True
    enable_text_special_features, Type: Categorical, Choices: {True, False}, Default: True
    prepro_analyzer, Type: Categorical, Choices: {word, char, char_wb}, Default: word
    prepro_max_features, Type: UniformInteger, Range: [5, 20000], Default: 10000, on log-scale
    prepro_min_df, Type: UniformFloat, Range: [0.005, 1.0], Default: 0.2, on log-scale

In [76]:
def rows_from_configspace(cs, model: str) -> list[HPRow]:
    """
    Convert a ConfigSpace.ConfigurationSpace into HPRow rows.
    Supports common HP types: categorical, uniform int/float (with log).
    """
    rows: list[HPRow] = []

    for hp in cs.get_hyperparameters():
        # name
        name = getattr(hp, "name", str(hp))

        # default (you can ignore in LaTeX later)
        default = getattr(hp, "default_value", "")

        # categorical
        if hasattr(hp, "choices"):
            domain = "{" + ", ".join(str(c) for c in hp.choices) + "}"
            rows.append(
                HPRow(
                    model=model,
                    hp_name=name,
                    hp_type="categorical",
                    default=str(default),
                    domain=domain,
                    log="no",
                )
            )
            continue

        # numeric (UniformInteger/UniformFloat, etc.)
        lower = getattr(hp, "lower", None)
        upper = getattr(hp, "upper", None)
        log = bool(getattr(hp, "log", False))

        if lower is not None and upper is not None:
            hp_type = "uniform_int" if isinstance(lower, int) and isinstance(upper, int) else "uniform_float"
            domain = f"[{lower}, {upper}]"
            rows.append(
                HPRow(
                    model=model,
                    hp_name=name,
                    hp_type=hp_type,
                    default=str(default),
                    domain=domain,
                    log="yes" if log else "no",
                )
            )
            continue

        # fallback (rare types): stringify
        rows.append(
            HPRow(
                model=model,
                hp_name=name,
                hp_type=hp.__class__.__name__,
                default=str(default),
                domain=str(hp),
                log="no",
            )
        )

    return rows


In [77]:
rows = rows_from_configspace(cs, "Preprocessing")
tex = to_latex_longtable(rows)
print(tex)

\small
\setlength{\tabcolsep}{3pt}
\begin{longtable}{l l l c p{5.4cm}}
\caption{Hyperparameter search spaces.}\label{tab:search_spaces}\\
\toprule
Model & Parameter & Type & Log & Range \\
\midrule
\endfirsthead
\toprule
Model & Parameter & Type & Log & Range \\
\midrule
\endhead
Preprocessing & allow\_nans & categorical & \ding{55} & \{True,\linebreak[1] \,\linebreak[1] False\} \\
 & enable\_categorical\_features & categorical & \ding{55} & \{True,\linebreak[1] \,\linebreak[1] False\} \\
 & enable\_datetime\_features & categorical & \ding{55} & \{True,\linebreak[1] \,\linebreak[1] False\} \\
 & enable\_raw\_text\_features & categorical & \ding{55} & \{True,\linebreak[1] \,\linebreak[1] False\} \\
 & enable\_text\_ngram\_features & categorical & \ding{55} & \{True,\linebreak[1] \,\linebreak[1] False\} \\
 & enable\_text\_special\_features & categorical & \ding{55} & \{True,\linebreak[1] \,\linebreak[1] False\} \\
 & prepro\_analyzer & categorical & \ding{55} & \{word,\linebreak[1] \,\l

C:\Users\poehlmann\AppData\Local\Temp\ipykernel_30480\336398005.py:8: DeprecationWarning: Please use `list(space.values())`
  for hp in cs.get_hyperparameters():
